In [2]:
import os
import sys
import glob
import pandas as pd

#avoid import errors
parent_dir=os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from scripts.CDS_machine import CDS_2_gffcomp
from scripts.statsCompression import gffstats_2data
import scripts.get_busco_db as bob


In [3]:
raw_data="../../data/species"
query_email=""
busco_database="../../data/busco_downloads/"

!ls $raw_data

Babesia_duncani_323732
Chaetoceros_neogracilis_240364
Chlamydomonas_reinhardtii_3055
Cladocopium_goreaui_2562237
Conticribra_weissflogii_1577725
Cryptosporidium_parvum_Iowa_II_353152
Cyanidiococcus_yangmingshanensis_2690220
Cyanidioschyzon_merolae_strain_10d_280699
Cyanidioschyzon_merolae_strain_10D_280699
Cyanidium_caldarium_2771
Cylindrotheca_closterium_2856
Dunaliella_salina_3046
Durusdinium_trenchii_1381693
Eimeria_necatrix_51315
Emiliania_huxleyi_CCMP1516_280463
Entamoeba_histolytica_HM-1:IMSS_294381
Galdieria_yellowstonensis_3028027
Giardia_duodenalis_5741
Giardia_muris_5742
Haematococcus_lacustris_44745
Hamiltosporidium_tvaerminnensis_1176355
Leishmania_donovani_5661
Leishmania_infantum_JPCM5_435258
Micromonas_pusilla_CCMP1545_564608
Naegleria_gruberi_5762
Paramecium_bursaria_74790
Paramecium_tetraurelia_5888
Plasmodium_berghei_ANKA_5823
Plasmodium_falciparum_3D7_36329
Plasmodium_vivax_5855
Plasmodium_yoelii_5861
Symbiodinium_natans_878477
Symbiodinium_necroappetens_1628268
Tetr

# Merge reference and predicted annotations

In [4]:
#use agat, as there are no in-python tools that do all the conflict resolution

In [ ]:
tr_sp=!ls $raw_data
pred_items = os.listdir("../results/pred") #only run on predicted items
pred_items=[tri.replace("_own.gff3", "") for tri in pred_items]
tr_sp = [species for species in tr_sp if species in pred_items]
print(len(tr_sp))


for src in ["own", "git"]:
    #checker(if things in file)
    fileWritten=False
    filepath=f"../job_commands/mergeAnn_{src}.txt"
    with open(filepath, "w") as f:
        print("mkdir -p ../results/merged", file=f)
        for sp in tr_sp:
            try:
                if src=="git":
                    #if no github parameters exist, dont process this sspecies_source busco evaluation
                    filename_git=f"../training_data/geneid-parameter-files/parameter_files/{sp}*.param"
                    found_files=glob.glob(filename_git) #supress error output

                    if not found_files:
                        raise FileNotFoundError(f"There is no github parameters for this species")
                
                fileWritten=True #if any species has git parameters the job command files will be written
                
                #reference annotation
                RefAnn=!realpath ../training_data/species/$sp/CDS_*.gff
                RefAnn=RefAnn[0]

                #annotation prediction by source (own or git)
                pred=!realpath ../results/pred/$sp*_$src*.gff3
                pred=pred[0]

                #create folder to store outputs
                merged_res=f"../results/merged/{sp}_{src}.gff" #protein sequence output

                #write agat commands
                agat_merge=f"agat_sp_merge_annotations.pl --gff {RefAnn} --gff {pred} --out {merged_res}"

                print(agat_merge)
                print(agat_merge, file=f)

            except Exception as e:
                print(f"{sp} presents error: {e}")
                continue
            
    if not fileWritten: #if the file is not written, delete it
        os.remove(filepath)
            

FileNotFoundError: [Errno 2] No such file or directory: '../results/pred'

In [ ]:
!ls ../results/merged

1-GeneIDTrain.ipynb	  3-GeneIDPredict.ipynb
2-GeneIDGitCompare.ipynb  __pycache__


# Prediction analysis

## Reference annotation(CDS) and Prediction comparison

## BUSCO assessment

In [ ]:
#agat for gff>prot
tr_sp=!ls $raw_data
pred_items = os.listdir("../results/merged")
pred_items=[tri.replace("_own.gff", "") for tri in pred_items]
tr_sp = [species for species in tr_sp if species in pred_items]
print(len(tr_sp))


for src in ["own", "git"]:
    #checker(if things in file)
    fileWritten=False
    filepath=f"../job_commands/M_buscoComp_{src}.txt"
    with open(filepath, "w") as f:
        print('cpus="${SLURM_CPUS_PER_TASK:-$(nproc)}"', file=f)
        for sp in tr_sp:
            try:
                if src=="git":
                    #if no github parameters exist, dont process this sspecies_source busco evaluation
                    filename_git=f"../training_data/geneid-parameter-files/parameter_files/{sp}*.param"
                    found_files=glob.glob(filename_git) #supress error output

                    if not found_files:
                        raise FileNotFoundError(f"There is no github parameters for this species")
                
                fileWritten=True #if any species has git parameters the job command files will be written
                
                #reference sequence
                RefSeq=!realpath ../training_data/species/$sp/CLEAN_*.fna
                RefSeq=RefSeq[0]

                #merged annotation by source (own or git)
                merged=!realpath ../results/merged/$sp*_$src*.gff
                merged=merged[0]

                #create folder to store outputs
                prot_res=f"../results/specie_logs/{sp}/{sp}_merged_prot.fa" #merged protein sequence output
                busco_outPath=f"../results/merged_busco/" #busco comparisons output
                busco_res=f"{sp}_busco" #busco out name

                #get species taxon
                taxID=sp[sp.rfind("_")+1:]
                #find lineage
                busco_lineage=bob.taxon2lineage(taxID, query_email, f"{busco_database}/file_versions.tsv", "odb12")

                #gff to protein command
                TOprotein_cmd=f"agat_sp_extract_sequences.pl \
                                -f {RefSeq} \
                                -g {merged} \
                                -t CDS \
                                -p \
                                -o {prot_res}"
                TOprotein_cmd=f"gffread -g {RefSeq} {merged} -y {prot_res}"

                print(TOprotein_cmd)
                print(TOprotein_cmd, file=f)
                
                busco_cmd=f"busco -m protein \
                            -i {prot_res} \
                            --download_path {busco_database} \
                            -l {busco_lineage} \
                            --cpu $cpus \
                            -f \
                            --opt-out-run-stats \
                            --out_path {busco_outPath} \
                            -o {busco_res}" #--offline #autolineage creates errors
                busco_cmd=busco_cmd.replace("                             ", " ")

                busco_plot=f"busco --opt-out-run-stats --plot {busco_outPath}{busco_res}"
                
                print(busco_cmd)
                print(busco_cmd, file=f)
                
                print(busco_plot)
                print(busco_plot, file=f)

            except Exception as e:
                print(f"{sp} presents error: {e}")
                continue
            
    if not fileWritten: #if the file is not written, delete it
        os.remove(filepath)
            